In [1]:
import os
import sys
import logging
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.milvus import MilvusVectorStore
from llama_index.core.node_parser import SentenceSplitter
from pymilvus import connections, utility
from llama_index.core.settings import Settings

ModuleNotFoundError: No module named 'llama_index'

In [5]:
load_dotenv()

True

In [6]:
logging.basicConfig(level=logging.DEBUG, filename="vector_db_debug.log", filemode="w")
logger = logging.getLogger(__name__)

In [7]:

collection_name = "indian_bare_acts_v1"
milvus_host = os.getenv("MILVUS_HOST", "localhost")
milvus_port = os.getenv("MILVUS_PORT", "19530")
uri = f"tcp://{milvus_host}:{milvus_port}"

# Connect to Milvus
connections.connect("default", uri=uri)
logger.info(f" Connected to Milvus at {uri}")

# Check if collection exists
if not utility.has_collection(collection_name):
    logger.info(f"Collection '{collection_name}' will be created during indexing.")


In [8]:
openai_api_key = os.getenv("OPENAI_API_KEY")
llm = OpenAI(api_key=openai_api_key)

# Embedding Model
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"  # dim = 384
embed_model = HuggingFaceEmbedding(model_name=embed_model_name)


# Manually load tokenizer
splitter = SentenceSplitter(
    chunk_size=256,
    chunk_overlap=50,
)

# Updated Service Configuration via Settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.text_splitter = splitter

In [9]:
import shutil
from typing import List
import re
# LlamaIndex (v0.9.48.post4 style imports)
try:
    from llama_index.core.readers import SimpleDirectoryReader
    from llama_index.core.schema import Document
    from llama_index.core.node_parser import SentenceSplitter
    from llama_index.core.service_context import ServiceContext
except ImportError as e:
    logging.error("CRITICAL: Failed to import LlamaIndex components.")
    logging.error(f"Details: {e}")
    sys.exit(1)
def clean_text(text: str) -> str:
    """
    Cleans legal text extracted from PDFs by:
    - Normalizing line endings
    - Stripping leading/trailing spaces
    - Removing only unwanted lines like page numbers or visual dividers
    - Preserving legal formatting and sections
    """
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    # Normalize newlines
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # Remove lines that are likely page numbers (standalone digits)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # Remove horizontal divider lines (e.g. ___, ----)
    text = re.sub(r'^[-_~=]{3,}\s*$', '', text, flags=re.MULTILINE)

    # Strip leading/trailing whitespace from each line
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)

    # Collapse 3+ newlines into 2
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()
# Local cleaner (your general-purpose one)

# Setup logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logger = logging.getLogger(__name__)


def load_and_clean_documents(input_dir: str) -> List[Document]:
    """
    Loads and lightly cleans supported documents from a directory.
    Applies basic whitespace/symbol cleanup only (preserves all meaningful content).
    """
    if not os.path.isdir(input_dir):
        raise ValueError(f"Input path '{input_dir}' is not a directory.")

    logger.info(f"Loading documents from: {input_dir}")
    try:
        reader = SimpleDirectoryReader(
            input_dir=input_dir,
            recursive=False,
            required_exts=[".pdf", ".docx"]
        )
        raw_documents = reader.load_data()
        logger.info(f" Found {len(raw_documents)} raw document segments.")
    except Exception as e:
        logger.error(f" Failed to read documents: {e}")
        return []

    processed_documents = []

    for i, doc in enumerate(raw_documents):
        raw_text = doc.text or ""
        print(f"\n--- Raw Segment {i} ---")
        print(f"Raw Length: {len(raw_text)} characters")
        print(f"Raw Preview (repr):\n{repr(raw_text[:300])}...\n")

        if not raw_text.strip():
            logger.warning(f" Skipping empty document segment at index {i}.")
            continue

        try:
            cleaned_text = clean_text(raw_text)

            print(f"Cleaned Length: {len(cleaned_text)} characters")
            print(f"Cleaned Preview (repr):\n{repr(cleaned_text[:300])}...\n")
            print("type : ",type(cleaned_text))

            if not cleaned_text.strip():
                logger.warning(f" Cleaned  segment {i} is empty after cleaning. Skipping.")
                continue

            metadata = getattr(doc, "extra_info", {}) or {}
            doc_id = os.path.basename(metadata.get("file_path", f"doc_{i}")).replace(".", "_")

            cleaned_doc = Document(
                text=cleaned_text,
                extra_info=metadata,
                doc_id=doc_id
            )

            print({
                "doc_id": cleaned_doc.doc_id,
                "metadata": cleaned_doc.metadata,
                "text": cleaned_doc.text[:500]  # Avoid printing full text unless needed
            })

            processed_documents.append(cleaned_doc)
            logger.info(f" Processed segment {i} from file: {metadata.get('file_name', doc_id)}")

        except Exception as e:
            logger.error(f"Error processing document segment {i}: {e}")
            continue

    print("\n===Summary ===")
    print(f"Total Raw Segments: {len(raw_documents)}")
    print(f"Total Valid Cleaned Documents: {len(processed_documents)}\n")

    if not processed_documents:
        raise ValueError("All documents failed to clean properly or were empty.")

    logger.info(f"Finished processing. Total valid cleaned documents: {len(processed_documents)}")
    return processed_documents

In [10]:
# Load cleaned documents from ../data (assuming notebook is inside /notebook/)

input_dir = "../data"
processed_docs = load_and_clean_documents(input_dir)
# Sanity check: Print a preview of the first document





--- Raw Segment 0 ---
Raw Length: 1325 characters
Raw Preview (repr):
'1 \n \nTHE DESIGNS ACT, 2000 \n_______ \nARRANGEMENT OF SECTIONS \n_______ \nCHAPTER I \nPRELIMINARY \nSECTIONS \n1. Short title, extent and commencement. \n2. Definitions. \n \nCHAPTER II \nREGISTRATION OF DESIGNS \n3. Controller and other officers. \n4. Prohibition of registration of certain designs. \n5. Applica'...

Cleaned Length: 1261 characters
Cleaned Preview (repr):
'THE DESIGNS ACT, 2000\n\nARRANGEMENT OF SECTIONS\n\nCHAPTER I\nPRELIMINARY\nSECTIONS\n1. Short title, extent and commencement.\n2. Definitions.\n\nCHAPTER II\nREGISTRATION OF DESIGNS\n3. Controller and other officers.\n4. Prohibition of registration of certain designs.\n5. Application for registration of designs.'...

type :  <class 'str'>
{'doc_id': 'DESIGNS_ACT_2000_pdf', 'metadata': {'page_label': '1', 'file_name': 'DESIGNS_ACT_2000.pdf', 'file_path': '/mnt/c/Users/mzahm/OneDrive/Desktop/Projects/legalredliner/notebook/../data/DESIGNS_ACT_2

/tmp/ipykernel_28557/2953680046.py:93: DeprecationWarning: Call to deprecated function (or staticmethod) extra_info. ('extra_info' is deprecated, use 'metadata' instead.) -- Deprecated since version 0.12.2.
  metadata = getattr(doc, "extra_info", {}) or {}



--- Raw Segment 260 ---
Raw Length: 4408 characters
Raw Preview (repr):
'40 \n78. Powers of Controller to correct clerical errors, etc. —(1) Without prejudice to the provisions \ncontained in sections 57 and 59 as regards amendment of appli cations for patents or complete \nspecifications 1[or other documents relating thereto] and subject to the provisions of section 44, th'...

Cleaned Length: 4293 characters
Cleaned Preview (repr):
'78. Powers of Controller to correct clerical errors, etc. —(1) Without prejudice to the provisions\ncontained in sections 57 and 59 as regards amendment of appli cations for patents or complete\nspecifications 1[or other documents relating thereto] and subject to the provisions of section 44, the\nCont'...

type :  <class 'str'>
{'doc_id': 'PATENTS_ACT_1970_pdf', 'metadata': {'page_label': '40', 'file_name': 'PATENTS_ACT_1970.pdf', 'file_path': '/mnt/c/Users/mzahm/OneDrive/Desktop/Projects/legalredliner/notebook/../data/PATENTS_ACT_1970.pdf', 'file_type'

In [11]:
# 🔹 Chunk all processed documents using the splitter

# Collect all non-empty chunks into one list
chunked_texts = []
for idx, doc in enumerate(processed_docs):
    chunks = splitter.split_text(doc.text)
    non_empty = [chunk for chunk in chunks if chunk.strip()]
    chunked_texts.extend(non_empty)





In [12]:
# Generate embeddings
embeddings = embed_model.get_text_embedding_batch(chunked_texts, show_progress=True)

print(f"✅ Total chunks embedded: {len(embeddings)}")
print(f"Embedding shape: {len(embeddings[0])} dimensions")


Generating embeddings:   0%|                                                 | 0/5823 [00:00<?, ?it/s]/mnt/c/Users/mzahm/OneDrive/Desktop/Projects/legalredliner/venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Generating embeddings: 100%|██████████████████████████████████████| 5823/5823 [02:43<00:00, 35.62it/s]

✅ Total chunks embedded: 5823
Embedding shape: 384 dimensions


In [13]:
# --- Phase 2.5: Preview chunks and embeddings ---
print("\n🔍 Preview of Chunks and Embeddings:\n")

for i in range(min(3, len(chunked_texts))):  # Preview first 3
    print(f"🔢 Embedding (first 5 dims): {embeddings[i][:5]}\n")
    if embeddings:
        print(f"✅ Embedding shape: ({len(embeddings)}, {len(embeddings[0])})")




🔍 Preview of Chunks and Embeddings:

🔢 Embedding (first 5 dims): [-0.05987910181283951, 0.06488045305013657, -0.004806725773960352, -0.06474905461072922, -0.09231699258089066]

✅ Embedding shape: (5823, 384)
🔢 Embedding (first 5 dims): [-0.0782998725771904, 0.06297812610864639, -0.019173411652445793, -0.0938698798418045, -0.007281686179339886]

✅ Embedding shape: (5823, 384)
🔢 Embedding (first 5 dims): [-0.06734886765480042, 0.015536193735897541, -0.0645473375916481, -0.05103633180260658, -0.06471562385559082]

✅ Embedding shape: (5823, 384)


In [14]:
from pymilvus import Collection, connections, FieldSchema, CollectionSchema, DataType, utility

# --- 1. Connect to Milvus ---
# (Assuming you are already connected from previous steps)
# connections.connect("default", host="localhost", port="19530")

collection_name = "indian_bare_acts_v1"

# --- 2. Drop the existing collection to ensure a clean start ---
if utility.has_collection(collection_name):
    print(f"🗑️ Found existing collection '{collection_name}', dropping it.")
    utility.drop_collection(collection_name)

# --- 3. Define Collection Schema ---
# Your schema is correct.
fields = [
    FieldSchema(name="chunk_id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535) # Increased max_length for safety
]
schema = CollectionSchema(fields=fields, description="Legal text chunks for RAG")
print(f"✅ Schema defined for '{collection_name}'.")

# --- 4. Create the Collection ---
collection = Collection(name=collection_name, schema=schema)
print(f"✅ Collection '{collection_name}' created.")

# --- 5. Prepare and Validate Data ---
# The 'embeddings' variable is already a list of lists, so .tolist() is not needed.
# We just use the variables 'chunked_texts' and 'embeddings' directly.
print(f"\nValidating data...")
print(f"Number of embeddings: {len(embeddings)}")
print(f"Number of text chunks: {len(chunked_texts)}")

# --- 6. Insert Data into Milvus ---
print("\n🚀 Inserting data into Milvus...")
collection.insert([
    embeddings,
    chunked_texts
])
collection.flush()
print(f"✅ Data inserted successfully.")
print(f"Total entities in collection: {collection.num_entities}")

# --- 7. Create an Index for the Vector Field ---
# This is a crucial step for enabling fast vector searches.
print("\nCreating index for the 'embedding' field...")
index_params = {
    "metric_type": "L2",       # Or "IP" for Inner Product
    "index_type": "IVF_FLAT",  # A common and effective index type
    "params": {"nlist": 128}   # Number of clusters
}
collection.create_index(field_name="embedding", index_params=index_params)
print("✅ Index created successfully.")

# --- 8. Load the Collection into Memory for Searching ---
print("\nLoading collection into memory...")
collection.load()
print("✅ Collection loaded and ready for searching.")

🗑️ Found existing collection 'indian_bare_acts_v1', dropping it.
✅ Schema defined for 'indian_bare_acts_v1'.
✅ Collection 'indian_bare_acts_v1' created.

Validating data...
Number of embeddings: 5823
Number of text chunks: 5823

🚀 Inserting data into Milvus...
✅ Data inserted successfully.
Total entities in collection: 5823

Creating index for the 'embedding' field...
✅ Index created successfully.

Loading collection into memory...
✅ Collection loaded and ready for searching.


In [35]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from pymilvus import Collection, connections

# --- 1. Setup ---
# Connect to Milvus
connections.connect("default", host="localhost", port="19530")
collection_name = "indian_bare_acts_v1"

# Instantiate the SAME embedding model used for storing data
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Get your collection
collection = Collection(name=collection_name)
collection.load() # Make sure it's loaded into memory

# --- 2. Create a Sample Query ---
query_text = "What is the punishment for theft?"

# Create an embedding for the query
query_embedding = embed_model.get_query_embedding(query_text)

# --- 3. Search the Collection ---
# Define search parameters
search_params = {
    "metric_type": "L2",
    "params": {"nprobe": 10}, # nprobe is a performance/accuracy trade-off
}

# Perform the search
results = collection.search(
    data=[query_embedding],      # The query vector
    anns_field="embedding",      # The field to search in
    param=search_params,
    limit=3,                     # Number of results to return
    output_fields=["text"]       # The field to return
)

# --- 4. Print the Results ---
print(f"🔍 Query: {query_text}\n")
print("Top 3 most similar results:\n")

for i, hit in enumerate(results[0]):
    print(f"--- Result {i+1} (Distance: {hit.distance:.4f}) ---")
    print(hit.entity.get('text'))
    print("\n")

🔍 Query: What is the punishment for theft?

Top 3 most similar results:

--- Result 1 (Distance: 0.4836) ---
306.Whoever, being a clerk or servant, or being employed in the capacity of a clerk or
servant, commits theft in respect of any property in the possession of his master or employer,
shall be punished with imprisonment of either description for a term which may extend to
seven years, and shall also be liable to fine.
307.Whoever commits theft, having made preparation for causing death, or hurt, or
restraint, or fear of death, or of hurt, or of restraint, to any person, in order to the committing
of such theft, or in order to the effecting of his escape after the committing of such theft, or
in order to the retaining of property taken by such theft, shall be punished with rigorous
imprisonment for a term which may extend to ten years, and shall also be liable to fine.
Illustrations.
(a)A commits theft on property in Z’s possession; and while committing this theft, he
has a loaded 

In [18]:
## --- Phase 5: RAG Pipeline Setup (Corrected) ---

import os
from dotenv import load_dotenv
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.milvus import MilvusVectorStore
from llama_index.core import VectorStoreIndex

# --- .env loading ---
dotenv_path = '../.env'
load_dotenv(dotenv_path=dotenv_path)
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    print("❌ ERROR: GROQ_API_KEY not found. Check your .env file and path.")
else:
    print("✅ Environment variables loaded successfully.")

# 1. Initialize the Groq LLM
llm = Groq(model="mixtral-8x7b-32768", api_key=groq_api_key)
print("✅ LLM (Groq) initialized.")

# 2. Initialize the Embedding Model
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding Model (HuggingFace) initialized.")

# 3. Connect to your existing Milvus Vector Store
collection_name = "indian_bare_acts_v1"
vector_store = MilvusVectorStore(
    uri="http://localhost:19530",
    collection_name=collection_name,
    text_key="text",  # <-- THIS IS THE REQUIRED FIX
    dim=384,
    overwrite=False
)
print(f"✅ Connection to Milvus collection '{collection_name}' established.")

✅ Environment variables loaded successfully.
✅ LLM (Groq) initialized.
✅ Embedding Model (HuggingFace) initialized.
✅ Connection to Milvus collection 'indian_bare_acts_v1' established.


In [19]:
## --- Phase 6: Building and Testing the Retriever ---

# Create a LlamaIndex Index object from your vector store
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=embed_model)
print("✅ Vector Store Index created.")

# Build the retriever, configuring it to fetch the top 3 results
retriever = index.as_retriever(similarity_top_k=3)
print("✅ Retriever built successfully.")

# --- Test the Retriever ---
print("\n--- Testing Retriever ---")
test_query = "What are the provisions for bail?"
retrieved_nodes = retriever.retrieve(test_query)

print(f"🔍 Test Query: '{test_query}'")
print(f"Retrieved {len(retrieved_nodes)} nodes from Milvus.\n")

for i, node in enumerate(retrieved_nodes):
    print(f"--- Node {i+1} (Score: {node.score:.4f}) ---")
    print(node.get_text())
    print("-" * 20)

✅ Vector Store Index created.
✅ Retriever built successfully.

--- Testing Retriever ---
🔍 Test Query: 'What are the provisions for bail?'
Retrieved 3 nodes from Milvus.

--- Node 1 (Score: 0.5544) ---
(2) The presence of the informant or any person authorised by him shall be obligatory
at the time of hearing of the application for bail to the person under section 65 or
sub-section (2) of  section 70 of the Bharatiya Nyaya Sanhita, 2023.
(3) A High Court or Court of Session may direct that any person who has been
released on bail under this Chapter be arrested and commit him to custody.
484. (1) The amount of every bond executed under this Chapter shall be fixed with
due regard to the circumstances of the case and shall not be excessive.
(2) The High Court or the Court of Session may direct that the bail required by a police
officer or Magistrate be reduced.
485.
--------------------
--- Node 2 (Score: 0.6172) ---
(3) If the case so requires, the bond or bail bond shall also bind the p

In [20]:
## --- Phase 7: Building the Full Query Engine ---
from llama_index.core.query_engine import RetrieverQueryEngine

# The query engine orchestrates the entire retrieve -> synthesize process
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    llm=llm,
)

print("✅ Full RAG Query Engine is ready.")

✅ Full RAG Query Engine is ready.


In [21]:
## --- Phase 8: Running a Query and Getting a Final Answer ---

# Define your question
final_query = "What does the Indian Penal Code say about criminal conspiracy?"

# Run the query
response = query_engine.query(final_query)

# Print the synthesized response from Groq
print("❓ Query:")
print(final_query)
print("\n\n💬 Answer:")
print(str(response))

# Print the source nodes used for the answer for verification
print("\n\n📚 Sources Used:")
for i, node in enumerate(response.source_nodes):
    print(f"--- Source {i+1} (Similarity Score: {node.score:.4f}) ---")
    print(node.get_text())
    print("-" * 20)

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}